# Holistic Data Preparer

## Part A: Conceptual Foundation

**Q1a. What is Data Analysis?**
Data Analysis is the process of inspecting, cleaning, and transforming raw data to discover useful patterns and insights. It helps in making informed decisions instead of guessing.

**Q1b. How to Plan a Data Science Project?**
A Data Science project is planned using the CRISP-DM framework: Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, and Deployment. It is a circular, iterative process.

**Q1c. How to Frame a Machine Learning Problem?**
Framing an ML problem means clearly defining the Input (features), Output (target), and Metric (success measure) before building any model, so the right approach is chosen.

**Q2. Tensors — In-depth Explanation**
A tensor is an N-dimensional array of numbers used to represent data in ML/DL. Scalars (0D), vectors (1D), matrices (2D), and higher-dimensional arrays (3D+) are all tensors.

In [1]:
import numpy as np

scalar = np.array(5)
vector = np.array([1, 2, 3])
matrix = np.array([[1, 2], [3, 4]])
tensor3d = np.random.rand(2, 3, 4)

print(scalar.ndim, vector.ndim, matrix.ndim, tensor3d.ndim)
print(tensor3d.shape)

0 1 2 3
(2, 3, 4)


## Part B: Data Acquisition

In [2]:
import pandas as pd
import numpy as np
import json
import sqlite3

df = pd.read_csv('customer_credit_risk_dataset.csv')
df.head()

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag
0,CUST1000,59.0,Female,South,Secondary,Self-Employed,582391.106165,189837.600775,Car,788.754647,2,38,40.612776,2016-12-07 05:48:54,0
1,CUST1001,49.0,Male,West,Primary,Salaried,667650.497506,287991.250278,Business,571.980726,1,33,52.866031,2022-06-09 05:11:55,0
2,CUST1002,35.0,Female,East,Post-Graduate,Salaried,563722.919378,451837.259987,Other,826.673527,1,48,69.387665,2017-01-23 01:27:55,1
3,CUST1003,63.0,Female,North,Graduate,Self-Employed,501129.132429,317216.171276,Business,602.452012,0,32,13.001754,2019-08-19 20:18:09,0
4,CUST1004,28.0,Male,North,Post-Graduate,Salaried,349860.787402,740726.408399,Business,542.795449,1,48,21.027790,2017-04-19 18:33:34,0


In [3]:
df.to_csv('customer_transactions.csv', index=False)

meta_cols = ['customer_id', 'age', 'gender', 'region']
df[meta_cols].to_json('customer_metadata.json', orient='records')

conn = sqlite3.connect('repayment.db')
df[['customer_id', 'repayment_history']].to_sql('repayment_history', conn, if_exists='replace', index=False)
conn.close()

def fetch_economic_indicators():
    return {'inflation_rate': 5.4, 'repo_rate': 6.5, 'gdp_growth': 7.2}

api_data = fetch_economic_indicators()
api_data

{'inflation_rate': 5.4, 'repo_rate': 6.5, 'gdp_growth': 7.2}

In [4]:
csv_df = pd.read_csv('customer_transactions.csv')

with open('customer_metadata.json') as f:
    json_df = pd.DataFrame(json.load(f))

conn = sqlite3.connect('repayment.db')
sql_df = pd.read_sql('SELECT * FROM repayment_history', conn)
conn.close()

df = csv_df.copy()
for k, v in api_data.items():
    df[k] = v

df.shape

(1000, 18)

## Part C: Data Understanding & Cleaning

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        1000 non-null   str    
 1   age                950 non-null    float64
 2   gender             960 non-null    str    
 3   region             1000 non-null   str    
 4   education_level    1000 non-null   str    
 5   employment_type    960 non-null    str    
 6   annual_income      940 non-null    float64
 7   loan_amount        1000 non-null   float64
 8   loan_purpose       1000 non-null   str    
 9   credit_score       950 non-null    float64
 10  repayment_history  1000 non-null   int64  
 11  transaction_count  1000 non-null   int64  
 12  spending_ratio     1000 non-null   float64
 13  join_date          1000 non-null   str    
 14  default_flag       1000 non-null   int64  
 15  inflation_rate     1000 non-null   float64
 16  repo_rate          1000 non-null   f

In [6]:
df.describe(include='all')

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag,inflation_rate,repo_rate,gdp_growth
count,1000,950.000000,960,1000,1000,960,9.400000e+02,1.000000e+03,1000,950.000000,1000.000000,1000.000000,1000.000000,1000,1000.000000,1.000000e+03,1000.0,1.000000e+03
unique,1000,NaN,3,4,4,3,NaN,NaN,5,NaN,NaN,NaN,NaN,1000,NaN,NaN,NaN,NaN
top,CUST1000,NaN,Male,West,Graduate,Salaried,NaN,NaN,Home,NaN,NaN,NaN,NaN,2016-12-07 05:48:54,NaN,NaN,NaN,NaN
freq,1,NaN,519,277,402,580,NaN,NaN,233,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
mean,NaN,42.448421,NaN,NaN,NaN,NaN,6.598076e+05,3.090610e+05,NaN,650.591144,1.524000,39.906000,25.391511,NaN,0.198000,5.400000e+00,6.5,7.200000e+00
std,NaN,12.676778,NaN,NaN,NaN,NaN,4.919294e+05,2.553437e+05,NaN,88.860161,1.207013,6.390149,23.866275,NaN,0.398692,8.886228e-16,0.0,1.777246e-15
min,NaN,21.000000,NaN,NaN,NaN,NaN,1.500000e+05,3.402986e+04,NaN,382.195952,0.000000,20.000000,1.000000,NaN,0.000000,5.400000e+00,6.5,7.200000e+00
25%,NaN,32.000000,NaN,NaN,NaN,NaN,4.648158e+05,1.642544e+05,NaN,591.684228,1.000000,35.000000,7.152367,NaN,0.000000,5.400000e+00,6.5,7.200000e+00
50%,NaN,43.000000,NaN,NaN,NaN,NaN,5.998450e+05,2.445563e+05,NaN,648.748201,1.000000,40.000000,17.349810,NaN,0.000000,5.400000e+00,6.5,7.200000e+00
75%,NaN,53.000000,NaN,NaN,NaN,NaN,7.510703e+05,3.630038e+05,NaN,706.792413,2.000000,44.000000,37.114082,NaN,0.000000,5.400000e+00,6.5,7.200000e+00


In [7]:
df.isnull().sum()

customer_id           0
age                  50
gender               40
region                0
education_level       0
employment_type      40
annual_income        60
loan_amount           0
loan_purpose          0
credit_score         50
repayment_history     0
transaction_count     0
spending_ratio        0
join_date             0
default_flag          0
inflation_rate        0
repo_rate             0
gdp_growth            0
dtype: int64

In [8]:
from sklearn.impute import SimpleImputer

num_cols = ['age', 'annual_income', 'credit_score']
imputer_num = SimpleImputer(strategy='median')
df[num_cols] = imputer_num.fit_transform(df[num_cols])

cat_cols = ['employment_type']
imputer_cat = SimpleImputer(strategy='most_frequent')
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])

df.isnull().sum()

customer_id           0
age                   0
gender               40
region                0
education_level       0
employment_type       0
annual_income         0
loan_amount           0
loan_purpose          0
credit_score          0
repayment_history     0
transaction_count     0
spending_ratio        0
join_date             0
default_flag          0
inflation_rate        0
repo_rate             0
gdp_growth            0
dtype: int64

In [9]:
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])
df['gender'].value_counts()

gender
Male      559
Female    411
Other      30
Name: count, dtype: int64

In [10]:
missing_mask = df['annual_income'].isnull().astype(int)
df['annual_income_missing'] = missing_mask
sample_vals = df['annual_income'].dropna().sample(df['annual_income'].isnull().sum(), random_state=1).values
df.loc[df['annual_income'].isnull(), 'annual_income'] = sample_vals

In [11]:
from sklearn.impute import KNNImputer

knn_cols = ['annual_income', 'loan_amount', 'credit_score']
knn_imputer = KNNImputer(n_neighbors=5)
df[knn_cols] = knn_imputer.fit_transform(df[knn_cols])

In [12]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_imputer = IterativeImputer(random_state=42)
df[knn_cols] = mice_imputer.fit_transform(df[knn_cols])

In [13]:
df_complete_case = df.dropna()
df_complete_case.shape

(1000, 19)

## Part D: Outlier Handling

In [14]:
from scipy import stats

z_scores = np.abs(stats.zscore(df['annual_income']))
df_zscore = df[z_scores < 3]
df_zscore.shape

(983, 19)

In [15]:
Q1 = df['loan_amount'].quantile(0.25)
Q3 = df['loan_amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df_iqr = df[(df['loan_amount'] >= lower) & (df['loan_amount'] <= upper)]
df_iqr.shape

(941, 19)

In [16]:
lower_p = df['credit_score'].quantile(0.01)
upper_p = df['credit_score'].quantile(0.99)
df['credit_score'] = df['credit_score'].clip(lower_p, upper_p)

In [17]:
from scipy.stats.mstats import winsorize

df['annual_income'] = winsorize(df['annual_income'], limits=[0.02, 0.02])

## Part E: Feature Engineering

In [18]:
df['join_date'] = pd.to_datetime(df['join_date'])
df['join_year'] = df['join_date'].dt.year
df['join_month'] = df['join_date'].dt.month
df['join_day'] = df['join_date'].dt.day
df['join_weekday'] = df['join_date'].dt.dayofweek

In [19]:
from sklearn.preprocessing import OrdinalEncoder

edu_order = [['Primary', 'Secondary', 'Graduate', 'Post-Graduate']]
ord_enc = OrdinalEncoder(categories=edu_order)
df['education_level_encoded'] = ord_enc.fit_transform(df[['education_level']])

In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['gender'])

In [21]:
df = pd.get_dummies(df, columns=['region', 'loan_purpose'], prefix=['region', 'purpose'])
df.head()

,customer_id,age,gender,education_level,employment_type,annual_income,loan_amount,credit_score,repayment_history,transaction_count,...,gender_encoded,region_East,region_North,region_South,region_West,purpose_Business,purpose_Car,purpose_Education,purpose_Home,purpose_Other
0,CUST1000,59.0,Female,Secondary,Self-Employed,582391.106165,189837.600775,788.754647,2,38,...,0,False,False,True,False,False,True,False,False,False
1,CUST1001,49.0,Male,Primary,Salaried,667650.497506,287991.250278,571.980726,1,33,...,1,False,False,False,True,True,False,False,False,False
2,CUST1002,35.0,Female,Post-Graduate,Salaried,563722.919378,451837.259987,826.673527,1,48,...,0,True,False,False,False,False,False,False,False,True
3,CUST1003,63.0,Female,Graduate,Self-Employed,501129.132429,317216.171276,602.452012,0,32,...,0,False,True,False,False,True,False,False,False,False
4,CUST1004,28.0,Male,Post-Graduate,Salaried,349860.787402,740726.408399,542.795449,1,48,...,1,False,True,False,False,True,False,False,False,False


In [22]:
df['income_bin'] = pd.cut(df['annual_income'], bins=5,
                          labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

df['credit_score_flag'] = (df['credit_score'] > 700).astype(int)

df['income_quantile_bin'] = pd.qcut(df['annual_income'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


In [23]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['transaction_count_bin'] = kmeans.fit_predict(df[['transaction_count']])

## Part F: Feature Scaling

In [24]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, Normalizer

scale_cols = ['annual_income', 'loan_amount']

std_scaler = StandardScaler()
df[[c + '_std' for c in scale_cols]] = std_scaler.fit_transform(df[scale_cols])

minmax_scaler = MinMaxScaler()
df[[c + '_minmax' for c in scale_cols]] = minmax_scaler.fit_transform(df[scale_cols])

maxabs_scaler = MaxAbsScaler()
df[[c + '_maxabs' for c in scale_cols]] = maxabs_scaler.fit_transform(df[scale_cols])

robust_scaler = RobustScaler()
df[[c + '_robust' for c in scale_cols]] = robust_scaler.fit_transform(df[scale_cols])

normalizer = Normalizer()
df[[c + '_norm' for c in scale_cols]] = normalizer.fit_transform(df[scale_cols])

df.head()

,customer_id,age,gender,education_level,employment_type,annual_income,loan_amount,credit_score,repayment_history,transaction_count,...,annual_income_std,loan_amount_std,annual_income_minmax,loan_amount_minmax,annual_income_maxabs,loan_amount_maxabs,annual_income_robust,loan_amount_robust,annual_income_norm,loan_amount_norm
0,CUST1000,59.0,Female,Secondary,Self-Employed,582391.106165,189837.600775,788.754647,2,38,...,-0.137248,-0.467147,0.395571,0.048052,0.515843,0.057939,-0.068865,-0.275315,0.950765,0.309914
1,CUST1001,49.0,Male,Primary,Salaried,667650.497506,287991.250278,571.980726,1,33,...,0.287125,-0.082557,0.489848,0.078323,0.591361,0.087896,0.267529,0.218542,0.918219,0.396074
2,CUST1002,35.0,Female,Post-Graduate,Salaried,563722.919378,451837.259987,826.673527,1,48,...,-0.230167,0.559433,0.374928,0.128854,0.499308,0.137902,-0.142521,1.042927,0.780289,0.625420
3,CUST1003,63.0,Female,Graduate,Self-Employed,501129.132429,317216.171276,602.452012,0,32,...,-0.541723,0.031954,0.305714,0.087336,0.443867,0.096815,-0.389486,0.365586,0.844945,0.534853
4,CUST1004,28.0,Male,Post-Graduate,Salaried,349860.787402,740726.408399,542.795449,1,48,...,-1.294651,1.691373,0.138447,0.217949,0.309884,0.226072,-0.986320,2.496462,0.427079,0.904214


## Part G: Feature Construction & Transformation

In [25]:
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log1p)
df['spending_ratio_log'] = log_transformer.transform(df[['spending_ratio']])

recip_transformer = FunctionTransformer(lambda x: 1 / (x + 1))
df['spending_ratio_recip'] = recip_transformer.transform(df[['spending_ratio']])

sqrt_transformer = FunctionTransformer(np.sqrt)
df['spending_ratio_sqrt'] = sqrt_transformer.transform(df[['spending_ratio']])

In [26]:
from sklearn.preprocessing import PowerTransformer

pt_boxcox = PowerTransformer(method='box-cox')
df['loan_amount_boxcox'] = pt_boxcox.fit_transform(df[['loan_amount']])

pt_yeo = PowerTransformer(method='yeo-johnson')
df['annual_income_yeo'] = pt_yeo.fit_transform(df[['annual_income']])

In [27]:
df['debt_to_income'] = df['loan_amount'] / df['annual_income']
df['avg_monthly_transactions'] = df['transaction_count'] / 6
df['spending_to_income'] = (df['spending_ratio'] / 100) * df['annual_income']

df[['debt_to_income', 'avg_monthly_transactions', 'spending_to_income']].head()

,debt_to_income,avg_monthly_transactions,spending_to_income
0,0.325962,6.333333,236525.194317
1,0.431350,5.500000,352960.319794
2,0.801524,8.000000,391154.172192
3,0.633003,5.333333,65155.575561
4,2.117203,8.000000,73567.992308


In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = ['age', 'annual_income', 'loan_amount']
categorical_features = ['employment_type']

numeric_pipeline = Pipeline([('scaler', StandardScaler())])
categorical_pipeline = Pipeline([('onehot', OrdinalEncoder())])

col_transformer = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

transformed_array = col_transformer.fit_transform(df)
transformed_array[:5]

array([[ 1.3379892 , -0.13724773, -0.4671473 ,  1.        ],
       [ 0.52826444,  0.28712489, -0.08255673,  0.        ],
       [-0.60535023, -0.23016732,  0.55943295,  0.        ],
       [ 1.66187911, -0.54172349,  0.03195383,  1.        ],
       [-1.17215757, -1.29465104,  1.69137295,  0.        ]])

## Part H: Final Deliverable

In [29]:
df.to_csv('final_processed_dataset.csv', index=False)
df.shape

(1000, 54)

### Report Summary

- **Missing values:** handled using Simple Imputer, Most Frequent Imputation, Missing Indicator + Random Sample, KNN Imputer, and MICE — numeric gaps filled with median/KNN-based estimates, categorical gaps with mode.
- **Outliers:** detected and treated in `annual_income`, `loan_amount`, `credit_score` using Z-score, IQR, Percentile clipping, and Winsorization.
- **Encoding:** Ordinal Encoding on `education_level`, Label Encoding on `gender`, One-Hot Encoding on `region` and `loan_purpose`.
- **Scaling/transformations applied and why:** Standardization, Min-Max, MaxAbs, Robust, and Normalizer applied to compare scale ranges; Log/Reciprocal/Sqrt and Box-Cox/Yeo-Johnson applied to fix skew in `spending_ratio`, `loan_amount`, `annual_income`.
- **Newly engineered features:** `debt_to_income`, `avg_monthly_transactions`, `spending_to_income`, plus date parts (`join_year/month/day/weekday`) and binned/binarized income and credit features.
- **Final dataset shape:** ready for ML modeling, saved as `final_processed_dataset.csv`.